# Method Selection and Sensitivity Analysis

**Session 128 Tutorial**

This notebook covers two critical aspects of applied causal inference:

1. **Method Selection**: How to choose the right estimator based on your data structure
2. **Sensitivity Analysis**: How to assess robustness to unmeasured confounding

## Learning Objectives

- Understand when each causal method is appropriate
- Apply E-values to assess robustness of observational findings
- Use Rosenbaum bounds for matched-pair sensitivity analysis
- Interpret and report sensitivity results

In [ ]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Visualization settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

np.random.seed(42)
print("Setup complete")

---
# Part 1: Method Selection Decision Tree

## When to Use Each Method

The choice of causal method depends on:
1. **Data structure** (RCT, observational, panel, etc.)
2. **Available instruments or design features**
3. **Assumptions you're willing to make**

## Decision Framework

```
What data structure do you have?

+-- RCT (randomized)
|   +-- Perfect compliance --> simple_ate()
|   +-- Noncompliance --> cace_2sls() (IV for CACE)
|   +-- Stratified randomization --> stratified_ate()
|
+-- Observational (no randomization)
|   +-- Good overlap, rich covariates --> dr_ate(), tmle_ate()
|   +-- Selection on observables --> psm_ate(), ipw_ate()
|   +-- Unconfoundedness questionable --> sensitivity analysis
|
+-- Policy change / Threshold
|   +-- Discontinuity at threshold --> SharpRDD(), FuzzyRDD()
|   +-- Kink at threshold --> SharpRKD()
|   +-- Policy timing varies --> callaway_santanna_ate()
|
+-- Instrument available
|   +-- Strong instrument (F>10) --> TwoStageLeastSquares()
|   +-- Weak instrument --> LIML(), Fuller()
|   +-- Multiple instruments --> GMM(), shift_share_iv()
|
+-- Treatment effect heterogeneity
    +-- Binary treatment --> s_learner(), t_learner(), x_learner()
    +-- Continuous treatment --> dml_continuous()
    +-- Distributional --> unconditional_qte(), conditional_qte()
```

## Method Comparison Table

| Method | Data Type | Key Assumption | Identifies |
|--------|-----------|----------------|------------|
| `simple_ate()` | RCT | Random assignment | ATE |
| `ipw_ate()` | Observational | Unconfoundedness + positivity | ATE |
| `psm_ate()` | Observational | Unconfoundedness + overlap | ATT |
| `dr_ate()` | Observational | Either propensity OR outcome model correct | ATE |
| `tmle_ate()` | Observational | Consistent nuisance estimation | ATE |
| `TwoStageLeastSquares()` | IV | Exclusion restriction | LATE |
| `SharpRDD()` | RDD | Continuity at threshold | LATE |
| `did_2x2()` | Panel | Parallel trends | ATT |

In [ ]:
# Generate data compatible with multiple methods
def generate_flexible_dgp(
    n: int = 1000,
    true_ate: float = 2.0,
    confounding: float = 0.5,
    seed: int = 42,
) -> dict:
    """Generate data that can be analyzed multiple ways.
    
    Model:
        X ~ N(0, I_5)
        propensity = expit(confounding * X[:,0])
        D ~ Bernoulli(propensity)  # Treatment
        Y = 1 + X[:,0] + X[:,1] + ATE*D + noise
        
    Returns dict with outcome, treatment, covariates, propensity, true_ate.
    """
    rng = np.random.default_rng(seed)
    
    # Covariates
    X = rng.normal(0, 1, (n, 5))
    
    # Propensity score (confounded)
    logit_p = confounding * X[:, 0]
    propensity = 1 / (1 + np.exp(-logit_p))
    
    # Treatment assignment
    D = rng.binomial(1, propensity, n)
    
    # Outcome (linear with confounding)
    baseline = 1 + X[:, 0] + 0.5 * X[:, 1]
    noise = rng.normal(0, 1, n)
    Y = baseline + true_ate * D + noise
    
    return {
        'outcome': Y,
        'treatment': D,
        'covariates': X,
        'propensity': propensity,
        'true_ate': true_ate,
    }

data = generate_flexible_dgp(n=2000, true_ate=2.0, confounding=0.5)
print(f"Generated {len(data['outcome'])} observations")
print(f"Treatment rate: {data['treatment'].mean():.1%}")
print(f"True ATE: {data['true_ate']}")

In [ ]:
# Demonstrate confounding visually
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Propensity score distribution by treatment
ax = axes[0]
ax.hist(data['propensity'][data['treatment'] == 0], bins=30, alpha=0.6, 
        label='Control', density=True)
ax.hist(data['propensity'][data['treatment'] == 1], bins=30, alpha=0.6, 
        label='Treated', density=True)
ax.set_xlabel('Propensity Score')
ax.set_ylabel('Density')
ax.set_title('Propensity Score Overlap')
ax.legend()

# Covariate imbalance
ax = axes[1]
X0_treated = data['covariates'][data['treatment'] == 1, 0]
X0_control = data['covariates'][data['treatment'] == 0, 0]
ax.hist(X0_control, bins=30, alpha=0.6, label='Control', density=True)
ax.hist(X0_treated, bins=30, alpha=0.6, label='Treated', density=True)
ax.set_xlabel('X[0] (Confounder)')
ax.set_ylabel('Density')
ax.set_title('Covariate Imbalance (Confounding)')
ax.legend()

plt.tight_layout()
plt.show()

print(f"SMD for X[0]: {(X0_treated.mean() - X0_control.mean()) / np.sqrt((X0_treated.var() + X0_control.var())/2):.3f}")

In [ ]:
# Import estimators
import sys
sys.path.insert(0, '/home/brandon_behring/Claude/causal_inference_mastery')

from src.causal_inference.rct import simple_ate
from src.causal_inference.observational import ipw_ate_observational, dr_ate
from src.causal_inference.psm import psm_ate

print("Estimators loaded successfully")

In [ ]:
# Compare methods on the same data
Y = data['outcome']
D = data['treatment']
X = data['covariates']

results = {}

# 1. Naive difference (ignores confounding)
naive_result = simple_ate(Y, D)
results['Naive (simple_ate)'] = {
    'ate': naive_result['estimate'],
    'se': naive_result['se'],
}

# 2. IPW (adjusts for confounding via propensity)
ipw_result = ipw_ate_observational(Y, D, X)
results['IPW'] = {
    'ate': ipw_result['estimate'],
    'se': ipw_result['se'],
}

# 3. PSM (matching on propensity score)
psm_result = psm_ate(Y, D, X)
results['PSM'] = {
    'ate': psm_result['estimate'],
    'se': psm_result['se'],
}

# 4. Doubly Robust (combines propensity and outcome model)
dr_result = dr_ate(Y, D, X)
results['DR (AIPW)'] = {
    'ate': dr_result['estimate'],
    'se': dr_result['se'],
}

# Display comparison
print(f"True ATE: {data['true_ate']:.3f}")
print("\n" + "="*60)
print(f"{'Method':<20} {'ATE':>10} {'SE':>10} {'Bias':>10} {'Covers':>10}")
print("="*60)

for name, res in results.items():
    ate, se = res['ate'], res['se']
    bias = ate - data['true_ate']
    ci_lower = ate - 1.96 * se
    ci_upper = ate + 1.96 * se
    covers = ci_lower < data['true_ate'] < ci_upper
    print(f"{name:<20} {ate:>10.3f} {se:>10.3f} {bias:>+10.3f} {str(covers):>10}")

In [ ]:
# Forest plot comparison
fig, ax = plt.subplots(figsize=(10, 5))

methods = list(results.keys())
y_pos = np.arange(len(methods))
ates = [results[m]['ate'] for m in methods]
ses = [results[m]['se'] for m in methods]

# Plot estimates with CIs
ax.errorbar(ates, y_pos, xerr=[1.96*s for s in ses], fmt='o', capsize=5, 
            markersize=8, color='steelblue', capthick=2)

# True ATE line
ax.axvline(x=data['true_ate'], color='red', linestyle='--', linewidth=2, 
           label=f'True ATE = {data["true_ate"]}')

ax.set_yticks(y_pos)
ax.set_yticklabels(methods)
ax.set_xlabel('Average Treatment Effect (95% CI)')
ax.set_title('Estimator Comparison on Observational Data')
ax.legend(loc='lower right')
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## Key Insight: Naive vs. Adjusted Estimates

Notice how:
- **Naive estimate** is biased (doesn't account for confounding)
- **IPW, PSM, DR** all adjust for confounding and recover approximately correct ATE
- **DR (AIPW)** is doubly robust: consistent if either propensity OR outcome model is correct

### When to Use Each:

| Situation | Recommended Method |
|-----------|-------------------|
| Good propensity model confidence | IPW |
| Limited overlap | PSM (focuses on comparable units) |
| Uncertain about either model | DR / TMLE (double robustness) |
| High-dimensional confounders | Machine learning + DML |

---
# Part 2: Sensitivity Analysis

## The Unconfoundedness Problem

All observational methods assume **no unmeasured confounders**:

$$Y(0), Y(1) \perp\!\!\!\perp D \mid X$$

This assumption is **untestable**. Sensitivity analysis asks:

> "How much unmeasured confounding would be needed to explain away our finding?"

## Two Approaches

1. **E-value** (VanderWeele & Ding 2017)
   - Works for any effect measure (RR, OR, HR, SMD)
   - Single interpretable number
   - Answers: "Minimum confounding strength to nullify result"

2. **Rosenbaum Bounds** (1987, 2002)
   - Specialized for matched studies (PSM)
   - Gamma (Γ) parameter: odds ratio of treatment assignment
   - Answers: "At what Γ does significance disappear?"

In [ ]:
# Import sensitivity analysis tools
from src.causal_inference.sensitivity import e_value, rosenbaum_bounds

print("Sensitivity analysis tools loaded")

## E-Value: Universal Sensitivity Metric

### Formula

For a risk ratio RR > 1:

$$E = RR + \sqrt{RR \times (RR - 1)}$$

### Interpretation

E-value = 3 means an unmeasured confounder would need:
- 3-fold association with treatment (RR_UD = 3)
- 3-fold association with outcome (RR_UY = 3)
- To fully explain away the observed effect

### Robustness Guidelines

| E-value | Interpretation |
|---------|---------------|
| ~1.0 | No robustness (effect is null) |
| ~1.5 | Weak robustness |
| ~2.0 | Moderate robustness |
| ~3.0 | Strong robustness |
| >4.0 | Very robust |

In [ ]:
# E-value for a risk ratio
# Example: Smoking and lung cancer (RR ~ 10)
result = e_value(10.0, ci_lower=8.0, ci_upper=12.5, effect_type='rr')

print("E-VALUE ANALYSIS: Smoking and Lung Cancer")
print("="*50)
print(f"Risk Ratio: 10.0 (95% CI: 8.0-12.5)")
print(f"")
print(f"E-value (point estimate): {result['e_value']:.2f}")
print(f"E-value (confidence limit): {result['e_value_ci']:.2f}")
print(f"")
print(result['interpretation'])

In [ ]:
# E-value for our DR estimate (convert to approximate RR)
# Use SMD approach for continuous outcomes

# First, compute standardized effect
pooled_sd = np.sqrt((Y[D==1].var() + Y[D==0].var()) / 2)
smd = dr_result['estimate'] / pooled_sd

print(f"DR estimate: ATE = {dr_result['estimate']:.3f}")
print(f"Pooled SD: {pooled_sd:.3f}")
print(f"Standardized Mean Difference (d): {smd:.3f}")

# Compute E-value using SMD
smd_ci_lower = (dr_result['estimate'] - 1.96 * dr_result['se']) / pooled_sd
smd_ci_upper = (dr_result['estimate'] + 1.96 * dr_result['se']) / pooled_sd

e_result = e_value(smd, ci_lower=smd_ci_lower, ci_upper=smd_ci_upper, effect_type='smd')

print(f"\nE-VALUE ANALYSIS: DR Estimate")
print("="*50)
print(f"E-value (point estimate): {e_result['e_value']:.2f}")
print(f"E-value (confidence limit): {e_result['e_value_ci']:.2f}")
print(f"RR equivalent: {e_result['rr_equivalent']:.2f}")

In [ ]:
# Visualize E-value interpretation
fig, ax = plt.subplots(figsize=(10, 6))

# Plot confounding region
rr_ud = np.linspace(1, 5, 100)
rr_uy = np.linspace(1, 5, 100)
RR_UD, RR_UY = np.meshgrid(rr_ud, rr_uy)

# Confounding that could explain away RR=2.0
observed_rr = e_result['rr_equivalent']

# Bounding factor (approximation)
# If RR_UD * RR_UY / (RR_UD + RR_UY - 1) >= observed_rr, could explain away
confounding_strength = (RR_UD * RR_UY) / (RR_UD + RR_UY - 1)

# Plot
contour = ax.contourf(RR_UD, RR_UY, confounding_strength, levels=20, cmap='RdYlGn_r')
plt.colorbar(contour, label='Confounding Strength')

# E-value point
e_val = e_result['e_value']
ax.plot(e_val, e_val, 'ko', markersize=15, label=f'E-value = {e_val:.2f}')

# Contour where confounding = observed RR
ax.contour(RR_UD, RR_UY, confounding_strength, levels=[observed_rr], colors='black', 
           linewidths=2, linestyles='--')

ax.set_xlabel('RR$_{UD}$ (Confounder-Treatment Association)')
ax.set_ylabel('RR$_{UY}$ (Confounder-Outcome Association)')
ax.set_title('E-value: Confounding Required to Explain Away Effect')
ax.legend(loc='upper right')
ax.set_xlim(1, 5)
ax.set_ylim(1, 5)

plt.tight_layout()
plt.show()

## Rosenbaum Bounds: Sensitivity for Matched Studies

### The Gamma Parameter

In a matched pair, if units are identical on observables, they should have equal treatment probability (Γ = 1).

With unmeasured confounding, one unit might be up to Γ times more likely to be treated:

$$\frac{1}{1+\Gamma} \leq P(T_i = 1) \leq \frac{\Gamma}{1+\Gamma}$$

### Interpretation

- Γ = 1: No unmeasured confounding
- Γ = 2: One unit could be twice as likely to be treated
- Γ* (critical): The smallest Γ where p-value exceeds α

### Robustness Guidelines

| Critical Γ | Interpretation |
|------------|---------------|
| 1.0-1.2 | Very sensitive |
| 1.5-2.0 | Moderately sensitive |
| 2.0-3.0 | Reasonably robust |
| >3.0 | Quite robust |

In [ ]:
# Generate matched-pair data for Rosenbaum bounds
def generate_matched_pairs(
    n_pairs: int = 100,
    true_effect: float = 1.5,
    seed: int = 42,
) -> tuple:
    """Generate matched pair outcomes.
    
    Within each pair:
    - Treated outcome: baseline + effect + noise
    - Control outcome: baseline + noise
    """
    rng = np.random.default_rng(seed)
    
    baseline = rng.normal(10, 2, n_pairs)  # Pair-level baseline
    noise_t = rng.normal(0, 1, n_pairs)
    noise_c = rng.normal(0, 1, n_pairs)
    
    treated = baseline + true_effect + noise_t
    control = baseline + noise_c
    
    return treated, control, true_effect

# Strong effect
treated_strong, control_strong, effect_strong = generate_matched_pairs(
    n_pairs=100, true_effect=2.0, seed=42
)

# Weak effect
treated_weak, control_weak, effect_weak = generate_matched_pairs(
    n_pairs=100, true_effect=0.3, seed=123
)

print(f"Strong effect: True effect = {effect_strong}")
print(f"  Observed mean difference: {(treated_strong - control_strong).mean():.3f}")
print(f"\nWeak effect: True effect = {effect_weak}")
print(f"  Observed mean difference: {(treated_weak - control_weak).mean():.3f}")

In [ ]:
# Rosenbaum bounds for strong effect
rb_strong = rosenbaum_bounds(
    treated_strong, 
    control_strong,
    gamma_range=(1.0, 4.0),
    n_gamma=30,
)

print("ROSENBAUM BOUNDS: Strong Effect")
print("="*50)
print(rb_strong['interpretation'])

In [ ]:
# Rosenbaum bounds for weak effect
rb_weak = rosenbaum_bounds(
    treated_weak, 
    control_weak,
    gamma_range=(1.0, 3.0),
    n_gamma=30,
)

print("ROSENBAUM BOUNDS: Weak Effect")
print("="*50)
print(rb_weak['interpretation'])

In [ ]:
# Visualize Rosenbaum bounds
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Strong effect
ax = axes[0]
ax.fill_between(rb_strong['gamma_values'], rb_strong['p_lower'], rb_strong['p_upper'],
                alpha=0.3, color='steelblue', label='P-value bounds')
ax.plot(rb_strong['gamma_values'], rb_strong['p_upper'], 'b-', linewidth=2, label='Upper bound')
ax.plot(rb_strong['gamma_values'], rb_strong['p_lower'], 'b--', linewidth=1, label='Lower bound')
ax.axhline(y=0.05, color='red', linestyle='--', label='α = 0.05')
if rb_strong['gamma_critical'] is not None:
    ax.axvline(x=rb_strong['gamma_critical'], color='green', linestyle=':', linewidth=2,
               label=f'Γ* = {rb_strong["gamma_critical"]:.2f}')
ax.set_xlabel('Γ (Sensitivity Parameter)')
ax.set_ylabel('P-value')
ax.set_title(f'Strong Effect (true = {effect_strong})')
ax.legend(loc='upper left')
ax.set_ylim(0, 0.5)
ax.grid(True, alpha=0.3)

# Weak effect
ax = axes[1]
ax.fill_between(rb_weak['gamma_values'], rb_weak['p_lower'], rb_weak['p_upper'],
                alpha=0.3, color='steelblue', label='P-value bounds')
ax.plot(rb_weak['gamma_values'], rb_weak['p_upper'], 'b-', linewidth=2, label='Upper bound')
ax.plot(rb_weak['gamma_values'], rb_weak['p_lower'], 'b--', linewidth=1, label='Lower bound')
ax.axhline(y=0.05, color='red', linestyle='--', label='α = 0.05')
if rb_weak['gamma_critical'] is not None:
    ax.axvline(x=rb_weak['gamma_critical'], color='green', linestyle=':', linewidth=2,
               label=f'Γ* = {rb_weak["gamma_critical"]:.2f}')
ax.set_xlabel('Γ (Sensitivity Parameter)')
ax.set_ylabel('P-value')
ax.set_title(f'Weak Effect (true = {effect_weak})')
ax.legend(loc='upper left')
ax.set_ylim(0, 0.5)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Comparing the Two Approaches

| Feature | E-value | Rosenbaum Bounds |
|---------|---------|------------------|
| **Scope** | Any observational study | Matched pairs only |
| **Input** | Effect estimate + CI | Pair outcomes |
| **Output** | Single number (E) | Curve (Γ vs p-value) |
| **Scale** | Risk ratio | Odds ratio |
| **Strengths** | Universal, easy to report | Design-specific, visual |

## Practical Guidance: Reporting Sensitivity Analysis

### E-value Reporting

> "The E-value for our point estimate (RR = 2.5) is 4.4, and for the confidence 
> interval limit (RR = 1.8) is 3.0. This means an unmeasured confounder would 
> need to be associated with both treatment and outcome by at least 3-fold to 
> reduce the observed effect to non-significance."

### Rosenbaum Bounds Reporting

> "Sensitivity analysis using Rosenbaum bounds found a critical Γ of 2.8. 
> This means the finding would remain significant (p < 0.05) unless matched 
> units differed in treatment probability by nearly 3-fold due to unmeasured 
> confounding."

### When to Worry

1. **E-value < 2** or **Γ* < 1.5**: Result is fragile
2. **Known unmeasured confounders**: Compare E-value to plausible effect sizes
3. **Weak instrument analogy**: If confounding of that strength is plausible in your domain, interpret cautiously

In [ ]:
# Final summary visualization
fig, ax = plt.subplots(figsize=(10, 6))

# Robustness spectrum
categories = ['Not Robust', 'Weakly\nRobust', 'Moderately\nRobust', 'Strongly\nRobust', 'Very\nRobust']
e_thresholds = [1.25, 1.75, 2.5, 4.0, 6.0]
colors = ['#d62728', '#ff7f0e', '#ffbb78', '#98df8a', '#2ca02c']

for i, (cat, threshold, color) in enumerate(zip(categories, e_thresholds, colors)):
    left = e_thresholds[i-1] if i > 0 else 1.0
    width = threshold - left
    ax.barh(0, width, left=left, height=0.4, color=color, alpha=0.8, edgecolor='black')
    ax.text(left + width/2, 0, cat, ha='center', va='center', fontsize=10, fontweight='bold')

# Mark our estimates
ax.plot(e_result['e_value'], 0, 'ko', markersize=15)
ax.annotate(f'DR estimate\nE = {e_result["e_value"]:.1f}', 
            xy=(e_result['e_value'], 0), xytext=(e_result['e_value'], 0.3),
            ha='center', fontsize=11,
            arrowprops=dict(arrowstyle='->', color='black'))

ax.set_xlim(1, 6)
ax.set_ylim(-0.5, 0.6)
ax.set_xlabel('E-value', fontsize=12)
ax.set_title('Sensitivity Analysis: Robustness Spectrum', fontsize=14)
ax.set_yticks([])
ax.spines['left'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
# Summary

## Method Selection

1. **Start with data structure**: RCT → simple methods; Observational → adjust for confounding
2. **Check overlap**: Poor overlap → PSM or trimming
3. **When uncertain**: Use doubly robust methods (DR, TMLE)
4. **Special designs**: IV for instruments, RDD for thresholds, DiD for policy changes

## Sensitivity Analysis

1. **Always report** sensitivity analysis for observational studies
2. **E-value**: Universal metric, easy to compute and interpret
3. **Rosenbaum bounds**: Best for matched designs, provides visual curve
4. **Interpretation**: Compare to plausible confounding in your domain

## References

- VanderWeele TJ, Ding P (2017). "Sensitivity Analysis in Observational Research: Introducing the E-Value."
- Rosenbaum PR (2002). "Observational Studies" (2nd ed.). Springer.
- Imbens GW, Rubin DB (2015). "Causal Inference for Statistics, Social, and Biomedical Sciences."